### 📄 Estrutura de Dados JSON – Documentação
Representação de uma entrada de vocabulário/frase para fins educacionais e linguísticos.

#### 🔁 Identificação e Control


| Campo        | Tipo                | Descrição                                                                 |
| ------------ | ------------------- | ------------------------------------------------------------------------- |
| `id`         | `string`            | Identificador único da entrada. Ideal para uso em banco de dados ou APIs. |
| `created_at` | `string (ISO 8601)` | Data e hora de criação do item. Formato UTC. Ex: `2025-05-19T14:30:00Z`.  |
| `updated_at` | `string (ISO 8601)` | Data e hora da última modificação do item.                                |
| `status`     | `string`            | Estado atual da entrada: `draft`, `published`, `pending review`, etc.     |
| `author`     | `string`            | Nome do autor ou sistema que criou/modificou o item.                      |


#### 📚 Conteúdo Linguístico

| Campo              | Tipo     | Descrição                                                           |
| ------------------ | -------- | ------------------------------------------------------------------- |
| `pergunta_pt-br`   | `string` | A frase original em português.                                      |
| `tradução_en`      | `string` | Tradução correspondente para o inglês.                              |
| `fonetica`         | `string` | Representação fonética simplificada da frase em inglês.             |
| `uso_da_linguagem` | `string` | Tipo de linguagem usada: informal, formal, técnico, etc.            |
| `difficulty_level` | `string` | Nível de dificuldade: `beginner`, `intermediate`, `advanced`.       |
| `part_of_speech`   | `string` | Classe gramatical: `noun`, `verb`, `expression`, `question`, etc.   |
| `category`         | `string` | Categoria geral da frase (ex: `viagem`, `negócios`, `cotidiano`).   |
| `subcategory`      | `string` | Subcategoria mais específica (ex: `pedir_instruções`, `alfândega`). |


#### 💬 Contexto e Exemplos

| Campo        | Tipo     | Descrição                                                        |
| ------------ | -------- | ---------------------------------------------------------------- |
| `context`    | `string` | Explicação de onde ou como a frase é usada.                      |
| `example_pt` | `string` | Exemplo de uso da frase em português.                            |
| `example_en` | `string` | Exemplo equivalente em inglês, com aplicação da frase traduzida. |


In [ ]:
import os

os.chdir("..")
os.chdir("..")

!dir

app	  docs	       pyproject.toml  requirements.txt  trash
database  poetry.lock  README.md       tests


In [ ]:
from app.toolkit.utils.data_loader import DataLoader
from datetime import datetime
import os
import json
import uuid

# Configurações
KIND = ""
BASE_CREATED_AT = "2025-04-11T11:17:38.000000Z"  # Data fixa de criação
data_path = os.path.join("database", "extract_data_video", "data", "extracted_data", "{kind}", "data_organize")
data_loader = DataLoader(base_path=data_path.format(kind=KIND))
data_all_words = data_loader.get_all_words()

# Utilitários
def generates_unique_id():
    return uuid.uuid4().hex

def now_iso():
    return datetime.now().isoformat(timespec='microseconds') + "Z"

def generate_json(
        pergunta_pt_br,
        uso_da_linguagem,
        tradução_en,
        fonetica,
        category,
        subcategory,
        created_at,
        updated_at
):
    return {
        "id": generates_unique_id(),
        "pergunta_pt-br": pergunta_pt_br,
        "uso_da_linguagem": uso_da_linguagem,
        "tradução_en": tradução_en,
        "fonetica": fonetica,
        "difficulty_level": "beginner",
        "part_of_speech": "question",
        "category": category,
        "subcategory": subcategory,
        "context": "",
        "example_en": "",
        "example_pt": "",
        "created_at": created_at,
        "updated_at": updated_at,
        "observations": "",
        "status": "pending",
        "author": "automation extracted from video lessons"
    }

# Processamento
for dict_data_word in data_all_words:
    path = dict_data_word.get("path")
    path_json = path / "text_v2.json"

    if not path_json.exists():
        print(f"Arquivo não encontrado: {path_json}")
        continue

    try:
        with open(path_json, "r", encoding="utf-8") as f:
            old_data = json.load(f)

        category = path.parts[-3]
        subcategory = path.parts[-2]

        new_data = generate_json(
            pergunta_pt_br=old_data.get("pergunta_pt-br", ""),
            uso_da_linguagem=old_data.get("uso_da_linguagem", ""),
            tradução_en=old_data.get("tradução_en", ""),
            fonetica=old_data.get("fonetica", ""),
            category=category,
            subcategory=subcategory,
            created_at=BASE_CREATED_AT,
            updated_at=now_iso()
        )

        with open(path_json, "w", encoding="utf-8") as f:
            json.dump(new_data, f, ensure_ascii=False, indent=2)

        # print(f"[✔] Atualizado: {path_json}")

    except Exception as e:
        print(f"[✖] Erro ao processar {path_json}: {e}")
